# Generador de Dataset Intermareal

Pipeline simplificado para generar datasets de imágenes Sentinel-2 con valores de marea asociados.

**Salida:** CSV con columnas `fecha`, `hora_utc`, `imagen_rgb`, `imagen_scl`, `marea_m`

---

## 0 · Instalación de dependencias

In [ ]:
# Solo ejecutar una vez por entorno
%pip install copernicusmarine openeo xarray geopandas rioxarray matplotlib contextily rasterio pyproj shapely scikit-image netcdf4 pyTMD pandas --quiet

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path("..").resolve()))

## 1 · Imports

In [ ]:
import sys
import os
from pathlib import Path
from datetime import datetime, timedelta
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import geopandas as gpd
import contextily as ctx
import openeo
import rasterio


# Importar módulos del paquete intertidal
from intertidal.geometry import GeometryProcessor
from intertidal.openeo_client import OpenEOClient
from intertidal.overpass import get_overpass_times
from intertidal.tidemodel import PyTMDTideModel
from intertidal.raster import RasterProcessor
from intertidal.notebook_compat import (
    CoordinateUtils,
    OpenEOManager,
    download_date_rgb,
    download_date_scl,
    tif_to_scl,
    compute_scl_stats,
    load_scl_stack,
    build_reference_map,
    compute_transition_cloud_stats,
    download_reference_map_openeo,
    load_reference_map_tif,
    evaluate_transition_cloud_coverage_openeo,
)

print("✓ Módulos cargados correctamente")

## 2 · Configuración del AOI 

**Cambiar aquí para generar dataset de otro sitio**

In [ ]:
# ════════════════════════════════════════════════════════════════════════════
# CONFIGURACIÓN — Cambiar estos parámetros según el sitio a estudiar
# ════════════════════════════════════════════════════════════════════════════

# Nombre del sitio (se usa en nombres de archivos y títulos)
#site_name = "Foz"
#site_name = "Villaviciosa"
site_name = "Santander"

# AOI — Área de Interés (polígono en formato DMS: grados°minutos'segundos"hemisferio)
# Ejemplo: Ría de Foz (pentágono cubriendo el estuario)
# aoi_dms = [
#     '43°34\'47.9"N 7°14\'50.9"W',   # NW — boca noroeste
#     '43°34\'09.0"N 7°13\'25.4"W',    # NE — boca noreste
#     '43°32\'59.2"N 7°14\'17.7"W',  # SE — interior sureste
#     '43°31\'41.5"N 7°15\'32.1"W',  # S  — cabecera estuario
#     '43°32\'24.00"N 7°17\'6.00"W',   # SW — interior suroeste
# ]

# aoi_dms = [
#     '43°32\'30.7"N 5°23\'45.4"W',   
#     '43°32\'13.4"N 5°21\'54.8"W',   
#     '43°28\'23.3"N 5°25\'01.7"W',   
#     '43°29\'31.9"N 5°26\'55.7"W',   
# ]

aoi_dms = [
    '43°30\'06.2"N 3°50\'48.9"W',   # NW
    '43°30\'06.2"N 3°42\'11.9"W',   # NE
    '43°23\'01.8"N 3°42\'11.9"W',   # SE
    '43°23\'01.8"N 3°50\'48.9"W',   # SW
]



# Punto de referencia para el modelo de mareas (centro del AOI)
# Si no se especifica, se calcula automáticamente como centroide del polígono
tide_location = None  # None = usar centroide automático, o tupla (lat, lon)

# Periodo temporal: últimos 5 años desde hoy
end_date = datetime.now()
start_date = end_date - timedelta(days=2*365)  # 2 años atrás
time_extent = [
    start_date.strftime("%Y-%m-%d"),
    end_date.strftime("%Y-%m-%d")
]

# Umbrales de filtrado de nubes
scl_bad_classes = [3, 8, 9, 10, 11]  # Clases SCL consideradas "malas"
ref_bad_fraction_threshold = 0.05     # 5% para construir reference map
transition_cloud_threshold = 0.1    # 20% para recuperar escenas válidas

# Directorios de salida (usa directorio actual)
output_dir = Path(".")  # Directorio actual (dataset_foz/)
rgb_dir = output_dir / "rgb"
scl_dir = output_dir / "scl"
csv_path = output_dir / f"dataset_{site_name.lower()}.csv"
reference_map_path = output_dir / "reference_map.tif"

# Crear directorios
rgb_dir.mkdir(parents=True, exist_ok=True)
scl_dir.mkdir(parents=True, exist_ok=True)

print("═" * 70)
print(f"  CONFIGURACIÓN DEL DATASET — {site_name}")
print("═" * 70)
print(f"Periodo:          {time_extent[0]} → {time_extent[1]}")
print(f"Directorio:       {output_dir.resolve()}")
print(f"CSV de salida:    {csv_path}")
print("═" * 70)

## 3 · Procesar geometría del AOI

In [ ]:
# Convertir AOI a formato utilizable
coords = CoordinateUtils()
polygon = coords.make_polygon(aoi_dms)
bbox = coords.bbox_from_polygon(polygon)

# Calcular punto para modelo de mareas (centroide si no se especificó)
if tide_location is None:
    gdf = gpd.GeoDataFrame(geometry=[polygon], crs="EPSG:4326")
    centroid = gdf.geometry.iloc[0].centroid
    tide_location = (centroid.y, centroid.x)  # (lat, lon)

print(f"AOI bbox: {bbox}")
print(f"Centroide: lat={tide_location[0]:.6f}, lon={tide_location[1]:.6f}")

# Visualizar AOI
fig, ax = plt.subplots(figsize=(10, 8))
gdf_wm = gpd.GeoDataFrame(geometry=[polygon], crs="EPSG:4326").to_crs("EPSG:3857")
b = gdf_wm.total_bounds
buffer = 3000
ax.set_xlim(b[0] - buffer, b[2] + buffer)
ax.set_ylim(b[1] - buffer, b[3] + buffer)
ctx.add_basemap(ax, crs="EPSG:3857", source=ctx.providers.Esri.WorldImagery, zoom=12)
gdf_wm.boundary.plot(ax=ax, color="red", linewidth=2, label="AOI")
ax.legend()
ax.set_title(f"Área de Interés — {site_name}", fontsize=13, fontweight="bold")
ax.axis("off")
plt.tight_layout()
plt.show()

## 4 · Conectar a OpenEO y obtener fechas disponibles

In [ ]:
# Conectar al backend de Copernicus Dataspace
manager = OpenEOManager()
manager.connect()
conn = manager.connection

print(f"✓ Conectado como: {conn.describe_account()['name']}")

In [ ]:
# Obtener todas las fechas disponibles de Sentinel-2 en el periodo
print(f"Consultando fechas disponibles en {time_extent[0]} → {time_extent[1]}...")
overpass_times = get_overpass_times(bbox, time_extent)
all_dates = sorted(overpass_times.keys())

print(f"\n✓ Encontradas {len(all_dates)} fechas disponibles")
print(f"  Primera: {all_dates[0]} a las {overpass_times[all_dates[0]].strftime('%H:%M:%S')} UTC")
print(f"  Última:  {all_dates[-1]} a las {overpass_times[all_dates[-1]].strftime('%H:%M:%S')} UTC")

## 5 · Construir Reference Map (filtro de nubes mejorado)

Este paso descarga un año completo de SCL y construye un mapa de referencia que identifica:
- Zonas de agua estable
- Zonas de tierra estable  
- Zona de transición (intermareal)

El reference map permite filtrar nubes solo en la zona que importa (el estuario).

In [ ]:
# Construir reference map usando OpenEO
print("Construyendo Reference Map...")
print("(Este proceso puede tardar 5-15 minutos)\n")

status_ref = download_reference_map_openeo(
    conn=conn,
    bbox=bbox,
    time_extent=time_extent,
    out_path=str(reference_map_path),
    bad_classes=scl_bad_classes,
    bad_fraction_threshold=ref_bad_fraction_threshold,
    stable_threshold=0.95,
    transition_buffer_pixels=10,
    force=False,  # Cambiar a True para forzar recálculo
)

print(f"\n✓ Reference map: {status_ref}")

# Cargar el reference map
reference_map, ref_transform, ref_crs = load_reference_map_tif(str(reference_map_path))
transition_mask = (reference_map == 0)

print(f"\nEstadísticas del reference map:")
print(f"  Transición:     {(reference_map==0).sum():>8,} px (zona intermareal)")
print(f"  Agua estable:   {(reference_map==1).sum():>8,} px")
print(f"  Tierra estable: {(reference_map==2).sum():>8,} px")

## 6 · Evaluar nubosidad en zona de transición

Para cada fecha del periodo, calcula el % de nubes **solo en la zona intermareal**.
Esto permite recuperar imágenes que tienen nubes sobre tierra firme pero el estuario despejado.

In [ ]:
print("Evaluando nubosidad en zona de transición...")
print("(Descargando cubo SCL del periodo completo)\n")

transition_stats_raw = evaluate_transition_cloud_coverage_openeo(
    conn=conn,
    bbox=bbox,
    time_extent=time_extent,
    reference_map=reference_map,
    reference_transform=ref_transform,
    bad_classes=scl_bad_classes,
    reference_dates=[],
    global_bad_fraction_threshold=ref_bad_fraction_threshold,
)

# Convertir a formato estándar
transition_stats = {
    date: {
        "bad_fraction": pct / 100,
        "bad_pct": pct,
    }
    for date, pct in transition_stats_raw.items()
}

print(f"\n✓ Evaluadas {len(transition_stats)} fechas")

## 7 · Filtrar fechas válidas

In [ ]:
# Clasificar fechas según el filtro de transición
valid_dates = sorted([
    d for d, s in transition_stats.items()
    if s["bad_fraction"] <= transition_cloud_threshold
])

discarded_dates = sorted([
    d for d, s in transition_stats.items()
    if s["bad_fraction"] > transition_cloud_threshold
])

# Subcategorías
reference_quality = [d for d in valid_dates if transition_stats[d]["bad_fraction"] <= ref_bad_fraction_threshold]
recovered = [d for d in valid_dates if transition_stats[d]["bad_fraction"] > ref_bad_fraction_threshold]

print("═" * 70)
print("  RESULTADO DEL FILTRADO")
print("═" * 70)
print(f"Total fechas evaluadas:        {len(transition_stats):>6}")
print(f"  ✓ Válidas:                   {len(valid_dates):>6}")
print(f"    - Calidad referencia (≤5%): {len(reference_quality):>5}")
print(f"    - Recuperadas (≤20%):       {len(recovered):>5}")
print(f"  ✗ Descartadas (>20%):        {len(discarded_dates):>6}")
print("═" * 70)

if recovered:
    improvement_pct = len(recovered) / len(valid_dates) * 100
    print(f"\n  El filtro de transición recuperó {len(recovered)} fechas adicionales")
    print(f"   ({improvement_pct:.1f}% del total de fechas válidas)")

## 8 · Descargar imágenes RGB y SCL de fechas válidas

⚠️ **Nota:** Esto puede tardar varias horas dependiendo del número de fechas.
Las descargas son idempotentes (se saltan archivos ya existentes).

In [ ]:
# Convertir polígono a formato GeoJSON para OpenEO
from shapely.geometry import mapping
polygon_geojson = mapping(polygon)

print(f"Descargando {len(valid_dates)} fechas válidas...\n")
print(f"Directorio RGB: {rgb_dir}")
print(f"Directorio SCL: {scl_dir}")
print(f"Recortando al polígono exacto del AOI\n")

download_results = {"rgb": {}, "scl": {}}

for i, date in enumerate(valid_dates, 1):
    cloud_pct = transition_stats[date]["bad_pct"]
    tag = "ref" if cloud_pct <= ref_bad_fraction_threshold * 100 else "✓"
    
    print(f"[{i:>4}/{len(valid_dates)}] {date} [{tag}] {cloud_pct:>5.1f}% nubes")
    
    # Descargar RGB (recortado al polígono)
    status_rgb = download_date_rgb(conn, date, bbox, str(rgb_dir), polygon=polygon_geojson)
    download_results["rgb"][date] = status_rgb
    
    # Descargar SCL (recortado al polígono)
    status_scl = download_date_scl(conn, date, bbox, str(scl_dir), polygon=polygon_geojson)
    download_results["scl"][date] = status_scl

print(f"\n✓ Descarga completada ({len(valid_dates)} fechas procesadas)")

## 9 · Calcular valores de marea

Para cada fecha válida, calcula la altura de marea en el momento exacto de adquisición de la imagen.

In [ ]:
print("Inicializando modelo de mareas...\n")

# Inicializar modelo de mareas (GOT4.10)
# Nota: box_size=2.0 permite buscar datos válidos en un área más amplia
# (útil en zonas costeras donde el modelo puede tener NaN cerca de tierra)
tide_model = PyTMDTideModel(
    model_name="GOT4.10",
    directory=str(output_dir / "tide_models"),
    box_size=2.0,  # Aumentado de 0.4 a 2.0 grados (~220 km de radio)
    resolution=0.05
)

print(f"✓ Modelo de mareas cargado: GOT4.10")
print(f"  Ubicación: lat={tide_location[0]:.6f}, lon={tide_location[1]:.6f}")
print(f"  Área de búsqueda: ±{2.0}° (~220 km de radio)\n")

# Calcular mareas y cobertura para todas las fechas válidas
tide_data = []

for date in valid_dates:
    # Obtener hora exacta de adquisición
    dt = overpass_times.get(date)
    if dt is None:
        print(f"⚠️  Sin hora de paso para {date}, saltando...")
        continue
    
    # Calcular cobertura del tile (% de píxeles válidos)
    cobertura_pct = 0.0
    tif_rgb_path = rgb_dir / f"rgb_{date}.tif"
    
    if tif_rgb_path.exists():
        try:
            with rasterio.open(tif_rgb_path) as src:
                rgb = np.dstack([src.read(i) for i in [1, 2, 3]])
                valid_pixels = ~np.isnan(rgb).any(axis=2)
                cobertura_pct = (valid_pixels.sum() / valid_pixels.size) * 100
        except Exception as e:
            print(f"⚠️  Error leyendo {date} para cobertura: {e}")
    
    # Calcular marea
    try:
        tide_m = tide_model.get_tide_height(
            lat=tide_location[0],
            lon=tide_location[1],
            dt=dt
        )
        
        tide_data.append({
            "fecha": date,
            "hora_utc": dt.strftime("%H:%M:%S"),
            "datetime_utc": dt.isoformat(),
            "imagen_rgb": f"rgb/rgb_{date}.tif",
            "imagen_scl": f"scl/scl_{date}.tif",
            "marea_m": round(tide_m, 3),
            "nubes_pct": round(transition_stats[date]["bad_pct"], 2),
            "cobertura_tile_pct": round(cobertura_pct, 2),
            "lat": tide_location[0],
            "lon": tide_location[1],
        })
    except Exception as e:
        print(f"⚠️  Error calculando marea para {date}: {e}")

print(f"\n✓ Calculadas mareas y cobertura para {len(tide_data)} fechas")

## 10 · Generar CSV del dataset

In [ ]:
# Validar que hay datos de marea calculados
if not tide_data:
    print("⚠️  Error: No hay datos de marea calculados.")
    print("   Asegúrate de ejecutar primero:")
    print("   - Celda 8: Descargar imágenes")
    print("   - Celda 9: Calcular valores de marea")
    raise ValueError("tide_data está vacío. Ejecuta las celdas 8 y 9 primero.")

# Crear DataFrame
df = pd.DataFrame(tide_data)

# Ordenar por fecha
df = df.sort_values("fecha").reset_index(drop=True)

# Guardar CSV con manejo de errores
try:
    df.to_csv(csv_path, index=False)
except PermissionError:
    print("\n⚠️  ERROR: No se puede guardar el CSV")
    print(f"   El archivo {csv_path.name} está abierto en otro programa")
    print("   → Cierra el archivo en Excel/editor y vuelve a ejecutar esta celda")
    raise

print("═" * 70)
print("  DATASET GENERADO")
print("═" * 70)
print(f"Archivo CSV:      {csv_path}")
print(f"Número de filas:  {len(df)}")
print(f"Columnas:         {', '.join(df.columns)}") 
print("═" * 70)

# Mostrar estadísticas de marea
print(f"\nEstadísticas de marea:")
print(f"  Mínima:  {df['marea_m'].min():>6.2f} m")
print(f"  Máxima:  {df['marea_m'].max():>6.2f} m")
print(f"  Media:   {df['marea_m'].mean():>6.2f} m")
print(f"  Mediana: {df['marea_m'].median():>6.2f} m")

# Mostrar estadísticas de cobertura del tile
print(f"\nEstadísticas de cobertura del tile:")
print(f"  Mínima:  {df['cobertura_tile_pct'].min():>6.2f}%")
print(f"  Máxima:  {df['cobertura_tile_pct'].max():>6.2f}%")
print(f"  Media:   {df['cobertura_tile_pct'].mean():>6.2f}%")

parciales = (df['cobertura_tile_pct'] < 95).sum()
if parciales > 0:
    print(f"  ⚠️  {parciales} imágenes con cobertura <95% (tiles parciales)")

# Mostrar primeras filas
print(f"\nPrimeras {min(10, len(df))} filas del dataset:\n")
print(df.head(10).to_string(index=False))

## 11 · Convertir a PNG normalizado (para modelos generativos)

Los archivos TIF contienen reflectancia cruda (0-10000). Para facilitar el entrenamiento de modelos generativos, convertimos a PNG con:
- Normalización a rango 0-255 (uint8)
- Contrast stretching (percentiles 2-98) para mejorar visualización
- Manejo de píxeles NaN (fuera del polígono)

In [ ]:
# Ejecutar conversión TIF → PNG usando el módulo intertidal.raster
df = RasterProcessor.convert_tifs_to_png(
    df=df,
    rgb_dir=rgb_dir,
    scl_dir=scl_dir,
    output_dir=output_dir,
    percentile_low=2,
    percentile_high=98,
    compress_level=6
)

# Guardar CSV actualizado
try:
    df.to_csv(csv_path, index=False)
    print(f"\n✓ CSV actualizado:")
    print(f"  - 'imagen_rgb' y 'imagen_scl' → PNG (para modelo generativo)")
    print(f"  - 'imagen_rgb_tif' y 'imagen_scl_tif' → TIF backup (georreferenciados)")
    print(f"  - 'cobertura_tile_pct' → % del AOI con datos válidos")
except PermissionError:
    print(f"\n⚠️  No se puede guardar CSV - archivo {csv_path.name} está abierto")
    print("   Ciérralo y ejecuta esta celda nuevamente")

# Opcional: Borrar TIF para ahorrar espacio (descomenta las siguientes líneas)
# import shutil
# shutil.rmtree(rgb_dir)
# shutil.rmtree(scl_dir)
# print(f"\n✓ Archivos TIF eliminados (PNG es suficiente para modelo generativo)")